# Make figure 11

Files needed:

Model data: not publicly available

Ocetrac output: `ocetrac-v9-blobs-tos-t1-r1-msq0-01860315-01891214-region.nc`

In [1]:
print('load libraries')

import xarray as xr
import numpy as np
import pandas as pd

import dask
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import cmocean as cm

import matplotlib.dates as mdates
import datetime
import cftime

import xwmb
import xwmt
import xgcm
import regionate
print(xwmb.__version__, xwmt.__version__, xgcm.__version__)

load libraries
0.6.0 0.2.0 0.9.0


In [2]:
model_data_path = "/dfs9/hfdrake_hpc/datasets/CM4_MHW_blobs/data_daily/"
ds = xr.open_mfdataset(f"{model_data_path}/*.ocean_daily.*.nc", chunks={"time":1})
# ds = ds.isel(yh=slice(1, None), yq=slice(None, -1), xh=slice(1,None), xq=slice(None, -1)) # realign cell center/corner coordinates

snap = xr.open_mfdataset(f"{model_data_path}/*.ocean_daily_snap*.nc", chunks={"time":1})
# Rename snapshot time coordinates to time_bounds so they can later be merged with ds
snap = snap.rename({
    **{'time': 'time_bounds'},
    **{v: f"{v}_bounds" for v in snap.data_vars}
    })

static = xr.open_dataset("/dfs9/hfdrake_hpc/datasets/CM4_MHW_blobs/data/WMT_monthly/ocean_month_rho2.static.nc")

path = "/pub/mariant3/WarmWaterMasses/data/"
labels = (xr.open_dataset(
    f"{path}ocetracv9/ocetrac-v9-blobs-tos-t1-r1-msq0-01860315-01891214-region.nc").sel(
    time=slice(f"0186", f"0186")).blobs.rename("event_mask"))

ds = xr.merge([ds.sel(time=ds.time[1:]), snap])
ds = xr.merge([static,ds],join='inner')

In [3]:
def add_estimated_layer_interfaces(ds):
    return ds.assign_coords({"zi": xr.DataArray(
        np.concatenate([[0], 0.5*(ds.zl.values[1:]+ds.zl.values[0:-1]), [6000]]),
        dims=('zi',)
    )})

ds = add_estimated_layer_interfaces(ds)

ds = ds.assign_coords({
    "areacello": xr.DataArray(ds["areacello"].values, dims=('yh', 'xh',)), # Required for area-integration
    "lon": xr.DataArray(ds["geolon"].values, dims=('yh', 'xh',)), # Required for calculating density if not already provided!
    "lat": xr.DataArray(ds["geolat"].values, dims=('yh', 'xh',)), # Required for calculating density if not already provided!
    "yq": xr.DataArray(ds["yq"].values, dims=('yq',)),
    "deptho": xr.DataArray(ds["deptho"].values, dims=('yh', 'xh',)),
    "geolon": xr.DataArray(ds["geolon"].values, dims=('yh', 'xh',)),
    "geolat": xr.DataArray(ds["geolat"].values, dims=('yh', 'xh',)),
    "geolon_c": xr.DataArray(ds["geolon_c"].values, dims=('yq', 'xq',)),
    "geolat_c": xr.DataArray(ds["geolat_c"].values, dims=('yq', 'xq',)),
    })

coords = {
    'X': {'center': 'xh', 'outer': 'xq'},
    'Y': {'center': 'yh', 'outer': 'yq'},
    'Z': {'center': 'zl', 'outer': 'zi'}
}

metrics = {
    ('X','Y'): "areacello", # Required for area-integration
    }

ds['tos'] = ds['thetao'].isel(zl=0)

lam = "heat"
grid = xgcm.Grid(ds.copy(), coords=coords, metrics=metrics, boundary={'X':'extend', 'Y':'extend', 'Z':'extend'}, autoparse_metadata=False)
wm = xwmt.WaterMass(grid)

In [4]:
import xbudget
budgets_dict = xbudget.load_preset_budget(model="MOM6_3Donly").copy()
del budgets_dict['salt']['lhs']
del budgets_dict['salt']['rhs']

# Note: the properties of this region are quite different from the rest of the Baltic!
name = "MANSO"
#lons = np.array([8.,   20.,  29., 24.5, 24.5, 26.1, 17.5, 11.5])
#lons = np.arange([-138, 0, 3.4])
lons = np.array([-137.,-120.,-100., -70., -70., -100., -120., -137.])
#lats = np.arange(8, 49, 1)
lats = np.array([10., 10., 10., 10., 38., 38., 38., 38.])
#lats = np.array([53.5, 53.5, 54.5,  59.,  61.,  63., 64.5,  62.])
manso_region = regionate.GriddedRegion(name, lons, lats, grid)

In [ ]:
lam = "heat"
with warnings.catch_warnings():
    warnings.simplefilter(action='ignore', category=FutureWarning)
    
    grid = xgcm.Grid(ds.copy(), coords=coords, metrics=metrics, boundary={'X':'extend', 'Y':'extend', 'Z':'extend'}, autoparse_metadata=False)
    wm = xwmt.WaterMass(grid)
    
    import xbudget
    budgets_dict = xbudget.load_preset_budget(model="MOM6_3Donly").copy()
    del budgets_dict['salt']['lhs']
    del budgets_dict['salt']['rhs']
    
    xbudget.collect_budgets(grid, budgets_dict)
    
    wmb = xwmb.WaterMassBudget(
        grid,
        budgets_dict,
        manso_region.mask
        )
    #display(wmb.grid._ds)
    wmb.mass_budget(lam, greater_than=True, default_bins=True)
    wmt = wmb.wmt


In [ ]:
# fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 11), sharex=True)
# plt.subplots_adjust(hspace=0.3)
dates_baja = [
    cftime.DatetimeNoLeap(186, 5, 29, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 8, 6, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 8, 12, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 8, 27, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 10, 27, 12, 0, 0, 0),
]
dates_gom = [
    cftime.DatetimeNoLeap(186, 4, 30, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 7, 24, 12, 0, 0),
    cftime.DatetimeNoLeap(186, 8, 9, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 9, 1, 12, 0, 0),
    cftime.DatetimeNoLeap(186, 10, 22, 12, 0, 0)
]

dates_pacific = [
    cftime.DatetimeNoLeap(186, 4, 12, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 5, 17, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 5, 24, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 5, 27, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 6, 17, 12, 0, 0, 0),
    cftime.DatetimeNoLeap(186, 6, 28, 12, 0, 0, 0)
]

In [ ]:
def get_region(event_id, ds, labels, region_name):
    """
    Extract vertical section at a predefined region location
    for the duration of a specific event.
    """

    # ---- Region coordinates ----
    regions = {
        "gom":     (-84,   27),
        "baja":    (-107,  19),
        "pacific": (-93.5, 15),
    }

    if region_name not in regions:
        raise ValueError(f"{region_name} not in {list(regions.keys())}")

    xh, yh = regions[region_name]

    # ---- Get event start/end ----
    ev = labels.where(labels == event_id, drop=True)

    if ev.time.size == 0:
        raise ValueError(f"Event {event_id} not found in labels")

    start = ev.time.isel(time=0)
    end   = ev.time.isel(time=-1)

    # ---- Extract vertical profile section ----
    region = (
        ds.thetao
        .sel(xh=xh, method="nearest")
        .sel(yh=yh, method="nearest")
        .sel(time=slice(start, end))
        .load()
    )

    return region, start.item(), end.item()

In [ ]:
def plot_subsurface(ds, ax, event_dates, vmin=25, vmax=33, depth_limit=25, pacific_labels=False):
    
    # vmin=26.5, vmax=31.6,
    1028.74475098
    import numpy as np
    import matplotlib.pyplot as plt
    import cmocean

    # Plot filled contours
    contourf = ds.plot.contourf(
        ax=ax,
        x="time", y="zl",
        cmap=cmocean.cm.balance,
        vmin=vmin, vmax=vmax,
        alpha=1, levels=35,
        add_colorbar=False
    )

    # Contour line at 29°C
    ds.plot.contour(
        ax=ax,
        x="time", y="zl",
        colors="k", levels=[29], linewidths=3, add_colorbar=False
    )

    # Add colorbar
    plt.colorbar(contourf, ax=ax, pad=0.01, ticks=np.arange(vmin, vmax + 0.1, 1)).set_label(r"$\Theta$ [deg C]")
    # plt.colorbar(contourf, ax=ax, pad=0.01, ticks=np.arange(vmin, vmax + 0.1, 0.5)).set_label( r"$\sigma_\theta$ [kg/m$^3$]")


    if pacific_labels:
        labels = ['Start of event', '1st appearance at surface', '1st disappearance at surface',
                  '2nd appearance at surface', '2nd disappearance at surface', 'End of event']
        colors = ['w', 'g', '#fa0202', 'g', '#fa0202', 'w']
        styles = ['--', '--', 'dashdot', '--', 'dashdot', '--']
        widths = [7, 4.5, 4.5, 4.5, 4.5, 7]
    else:
        labels = ['Start of event', '1st appearance at surface', 'Maximum surface area',
                  'Last appearance at surface', 'End of event']
        colors = ['w', 'g', '#fa0202', '#4103fc', 'w']
        styles = ['--', '--', 'dashdot', 'dotted', '--']
        widths = [7, 4.5, 4.5, 4.5, 4.5, 7]

    # Plot vertical lines for events
    for d, c, s, lw, label in zip(event_dates, colors, styles, widths, labels):
        ax.axvline(x=d, color=c, linestyle=s, linewidth=lw, label=label)

    # Format x-axis ticks and depth limit
    event_labels = [d.strftime("%Y-%m-%d") for d in event_dates]
    ax.set_xticks(event_dates)
    ax.set_xticklabels(event_labels, rotation=35, ha='right') #, fontsize=18)
    ax.set_ylim(depth_limit, 1)
    ax.set_ylabel("Depth [m]")
    ax.set_xlabel("")
    
    ax.set_title("")
    ax.legend(fontsize='small', loc=2, framealpha=0.8)


In [ ]:
regions = {
    "gom":     [-84,   27],
    "baja":    [-107,  19],
    "pacific": [-93.5, 15],
}

def get_region(ds, reg, t1, t2, var="thetao"):
    xh, yh = regions[reg]

    region = (
        ds[var]
        .sel(xh=xh, method="nearest")
        .sel(yh=yh, method="nearest")
        .sel(time=slice(t1, t2))
        .load()
    )

    return region

In [ ]:
ds_gom = get_region(
    ds=ds,
    reg="gom",
    t1="0186-04-30",
    t2="0186-10-22"
)

ds_baja = get_region(
    ds=ds,
    reg="baja",
    t1="0186-05-29",
    t2="0186-10-27"
)

ds_pacific = get_region(
    ds=ds,
    reg="pacific",
    t1="0186-04-12",
    t2="0186-06-28"
)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 21), sharex=False)
plt.subplots_adjust(hspace=0.25)

plot_subsurface(
    ds=ds_baja,
    ax=ax1,
    event_dates=dates_baja
)
ax1.text(0.008, 1.065, '(a) Event 559 in the Gulf of California',
         transform=ax1.transAxes, va='top',fontweight="bold")

plot_subsurface(
    ds=ds_gom,
    ax=ax2,
    event_dates=dates_gom
)
ax2.text(0.008, 1.065, '(b) Event 373 in the GoM',
         transform=ax2.transAxes, va='top',fontweight="bold")
ax2.legend().remove()
ax2.set_xlabel("")

plot_subsurface(
    ds=ds_pacific,
    ax=ax3,
    event_dates=dates_pacific,
    pacific_labels=True
)
ax3.text(0.008, 1.065, '(c) Event 252 in the Pacific Coast',
         transform=ax3.transAxes, va='top',fontweight="bold")
ax3.legend(fontsize='small', loc=2, framealpha=0.8)
ax3.set_xlabel("")

plt.tight_layout()

# plt.savefig("../figures/3-panel-subsurface-paper.png", dpi=400, bbox_inches="tight")
plt.show()